In [1]:
import litellm
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("OPENAI_API_KEY"):
    litellm.openai_key = os.getenv("OPENAI_API_KEY")

OPENAI_MODEL = os.getenv("OPENAI_MODEL")

In [3]:
import math, random, re, sys, platform, time, json
from typing import List, Dict
import numpy as np
import pandas as pd
from evaluate import load
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)



/Users/abhishekbarve/learning/ai-ml/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# exact match

preds = ['Apple', 'banana', 'Orange']
labels = ['apple', 'banana', 'orange']

def normalize(s: str) -> str:
    return s.lower().strip()

def exact_match(pred: str, label: str) -> int:
    return int(normalize(pred) == normalize(label))

em_scores = [exact_match(p, l) for p, l in zip(preds, labels)]

em_accuracy = sum(em_scores) / len(em_scores)

print(f'scores: {em_scores}, accuracy: {em_accuracy:.2f}')

scores: [1, 1, 1], accuracy: 1.00


In [7]:
# Lexical similarity ROGUE

pred = "the quick brown fox"
label = "the fox is quick and brown"

# Loading the ROUGE metric from the 'evaluate' library
rouge = load("rouge")

results = rouge.compute(predictions=[pred], references=[label])

# ROUGE-1 measures unigram (single word) overlap.
# ROUGE-L measures the longest common subsequence.
print(f"ROUGE-1 Score: {results['rouge1']:.4f}")
print(f"ROUGE-L Score: {results['rougeL']:.4f}")

ROUGE-1 Score: 0.8000
ROUGE-L Score: 0.6000


In [9]:
# Semantic similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

labels = ['A dog is a loyal pet', 'cats are independent animals', 'the sky is blue']

preds = [
    'dogs make great companions',
    'a cat is a solitary creature',
    'the ocean is vast',
]

pred_embeddings = model.encode(preds)
label_embeddings = model.encode(labels)

for i in range(len(preds)):
    similarity = np.dot(pred_embeddings[i], label_embeddings[i]) / (
        np.linalg.norm(pred_embeddings[i]) * np.linalg.norm(label_embeddings[i])
    )

    print(
        f'Pair {i + 1}: Pred: {preds[i]} Label: {labels[i]} Similarity: {similarity:.4f}'
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9207.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pair 1: Pred: dogs make great companions Label: A dog is a loyal pet Similarity: 0.6147
Pair 2: Pred: a cat is a solitary creature Label: cats are independent animals Similarity: 0.6848
Pair 3: Pred: the ocean is vast Label: the sky is blue Similarity: 0.3098


In [10]:
# functional correctness

def reverse_and_capitalize(s: str) -> str:
    if any(char.isdigit() for char in s):
        return 'Error contains digit'
    return s[::-1].upper()

code_preds = ['hello', 'world1', 'python']
test_labels = ['OLLEH', '1DLROW', 'NOHTYP']

results = []

for pred_code, label in zip(code_preds, test_labels):
    output = reverse_and_capitalize(pred_code)
    print(f'input: {pred_code} output: {output}, expected: {label}')
    results.append(output == label)

pass_rate = sum(results) / len(results)

print(f'pass rate: {pass_rate:.2f}')

input: hello output: OLLEH, expected: OLLEH
input: world1 output: Error contains digit, expected: 1DLROW
input: python output: NOHTYP, expected: NOHTYP
pass rate: 0.67


## 6. Pass@k

Sometimes we ask a model to generate multiple (`k`) possible answers for one problem. The **Pass@k** metric measures if at least one of these `k` attempts is correct. If any sample is correct, the entire set is considered a "pass" (score of 1).

In [12]:
def pass_at_k(samples: List[str], label: str) -> int:
    return int(any(s == label for s in samples))

label = 'blue'
samples = ['red', 'yellow', 'green', 'blue']

pass_score = pass_at_k(samples, label)

print(f'samples: {samples} \n label: {label} passscore: {pass_score}')

samples: ['red', 'yellow', 'green', 'blue'] 
 label: blue passscore: 1


## 7. LLM-as-a-Judge

For complex, subjective tasks (like creativity or helpfulness), we can use another powerful LLM to act as a judge. We provide the judge with the prediction, the reference (if any), and a detailed **rubric**. The judge then provides a score and reasoning.

In [15]:
RUBRIC = """
Score 1.0 if the predicted animal is the same as the label.
Score 0.5 if the prediction is a different animal but from the same biological class (e.g., both are mammals).
Score 0.0 otherwise (e.g., a mammal and a reptile).
"""

def llm_as_judge(pred: str, label: str, rubric: str) -> float:
    from litellm import completion
    import re

    response = completion(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert judge that scores answers based on a given rubric. "
                    "You respond with a reason, and then FINAL SCORE: <score>."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Rubric:\n{rubric}\n\nPrediction: {pred}\nLabel: {label}\n\nWhat score should be assigned"
                    " based on the rubric? Respond with scores in the rubric."
                ),
            },
        ]
    )
    response_text = response.choices[0].message.content

    print(f'llm judge response: {response_text}')

    pattern = r"FINAL SCORE:\s*([0-9]*\.?[0-9]+)"
    match = re.search(pattern, response_text)
    if match:
        score = float(match.group(1))
        return score
    else:
        print(
            "Warning: Could not find FINAL SCORE in the response. Defaulting to None."
        )
        return None

score1 = llm_as_judge(pred='Lion', label='Lion', rubric=RUBRIC)
print(f'final score: {score1}')

score2 = llm_as_judge(pred='Tiger', label='Lion', rubric=RUBRIC)
print(f'final score: {score2}')

score3 = llm_as_judge(pred='Snake', label='Lion', rubric=RUBRIC)
print(f'final score: {score3}')


llm judge response: Reason: The predicted animal (Lion) exactly matches the label (Lion), so it scores the highest possible.

FINAL SCORE: 1.0
final score: 1.0
llm judge response: Reason: The prediction (Tiger) and the label (Lion) are different species, but both are mammals, so they share the same biological class.

FINAL SCORE: 0.5
final score: 0.5
llm judge response: Reason: The predicted animal (snake) is a reptile, while the label (lion) is a mammal; different biological classes, so the score is 0.0.

FINAL SCORE: 0.0
final score: 0.0
